# TD: RAG

Dans ce notebook, un RAG basique est implémenté:
- On chunk les documents par paragraphes
- On a un embedding pour les chunks
- Pour une question, on peut embedde la question et récupérer les N chunks les plus pertinents
- On utilise un modèle de génération de texte (SMoLL) pour faire la partie question + chunks les plus pertinents -> réponse.

Téléchargez (cette archive)[https://drive.google.com/file/d/1TnfKs7bTwmpbXklbgiIBpdw7I_wJ5y9Y/view?usp=sharing] avec différentes 

Dans ce TD, vous allez expérimenter différentes façons de chunk et d'embeded les documents et les questions pour que le RAG retrieve les documents les plus pertinents. <br/>
Vous expérimenterez aussi la prompt donnée au générateur de texte pour avoir les meilleures réponses.

Voici la [liste de questions](https://drive.google.com/file/d/14hZ0hTx5dM1WgJYewZsn9BkHzEReq-pj/view?usp=sharing) que je poserai au RAG. </br>
A rendre: 
- Le notebook de votre RAG
- un CSV avec question,embedding,rag_reply
- un CSV avec chunk,embedding</br>
L'embedding doit être le JSON d'une liste de float.</br>
Quand je ferai "json.loads(embedding)", je dois récupérer une liste de floats

In [1]:
import numpy as np

import pandas as pd
from pathlib import Path

# Data loading

In [2]:
path = Path("../data/raw/rag/")

In [3]:
texts = []
for filename in path.glob("*.md"):
    with open(filename) as f:
        texts.append(f.read())

texts[0]

'# Title: Introduction to Cybersecurity: Principles and Practices  \n\n**Teacher:** Professor Lydia Carter  \n\n**Description:**  \nThis course introduces the fundamentals of cybersecurity, focusing on protecting systems, networks, and data from cyber threats. Students will explore key topics such as cryptography, network security, ethical hacking, and risk management. Through practical labs and real-world case studies, students will gain hands-on experience in identifying vulnerabilities, implementing security measures, and understanding the legal and ethical aspects of cybersecurity.  \n\n**Prerequisites:**  \n- Basic knowledge of computer networks and operating systems  \n- Proficiency in at least one programming language (e.g., Python, Java, or C++)  \n- Completion of "Introduction to Computer Science" or equivalent  \n\n**Assessment:**  \n- Weekly quizzes and assignments (25%)  \n- Midterm exam: Fundamentals of cybersecurity (20%)  \n- Final project: Design and present a comprehen

# Chunk
## Basic

In [4]:
def parse_class(text):
    chunks = text.split("\n\n")
    title = chunks[0].replace("# Title: ", "")
    return {"title": title, "chunks": chunks}

In [5]:
def parse_class_add_title(text):
    chunks = text.split("\n\n")
    title = chunks[0].replace("# Title: ", "")
    return {"title": title, "chunks": [f"{title}: {chunk}" for chunk in chunks]}

In [6]:
chunks = sum((parse_class_add_title(txt)["chunks"] for txt in texts), [])

# Embedding

## BAAI's embedding

In [7]:
%pip install -U FlagEmbedding

/home/arnaud/projects/old/nlp_esgi_2024/notebooks/.venv/bin/python3: No module named pip
Note: you may need to restart the kernel to use updated packages.


In [8]:
from FlagEmbedding import FlagModel

In [9]:
model = FlagModel(
    'BAAI/bge-base-en-v1.5',
    query_instruction_for_retrieval="Represent this sentence for searching relevant passages:",
    use_fp16=True,
)

In [10]:
corpus_embedding = model.encode(chunks)

You're using a BertTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


In [11]:
queries = [
    "Who is the reinforcement learning teacher?",
    "In what class will I learn game AI?",
]

In [12]:
query_embedding = model.encode(queries)

In [13]:
sim_scores = query_embedding @ corpus_embedding.T

In [14]:
for query, score in zip(queries, sim_scores):
    print(" ---- ")
    print("Query: ", query)
    indexes = np.argsort(score)[-5:]
    print("Sources:")
    for i, idx in enumerate(reversed(indexes)):
        if score[idx] > .5:
            print(f"{i+1} -- similarity {score[idx]:.2f} -- \"", chunks[idx], '"')
            
    

 ---- 
Query:  Who is the reinforcement learning teacher?
Sources:
1 -- similarity 0.80 -- " Foundations of Reinforcement Learning  : **Teacher:** Dr. Arjun Patel   "
2 -- similarity 0.74 -- " Foundations of Reinforcement Learning  : # Title: Foundations of Reinforcement Learning   "
3 -- similarity 0.71 -- " Foundations of Reinforcement Learning  : 2. **Tabular Methods**  
   - Dynamic programming approaches: Policy Iteration and Value Iteration  
   - Monte Carlo methods and Temporal-Difference (TD) Learning   "
4 -- similarity 0.71 -- " Foundations of Reinforcement Learning  : 4. **Policy-Based Methods**  
   - Policy Gradient methods and REINFORCE algorithm  
   - Advantage Actor-Critic (A2C) and Proximal Policy Optimization (PPO)   "
5 -- similarity 0.71 -- " Foundations of Reinforcement Learning  : **Description:**  
This course explores the foundational principles and practical applications of reinforcement learning (RL), a branch of machine learning focused on decision-making a

# Eval retrieval: Mean Reciprocal Rank
Le fichier [question_answer_short.csv](https://drive.google.com/file/d/1EB8IwGlqvpNy3oq7xyR2IzdqJDX8C_fr/view?usp=drive_link) contient une liste de question et le texte à retrouver dans les documents.<br/>
Je considère que tout chunk contenant le "texte à retrouver" était un bon chunk

In [15]:
df = pd.read_csv(path / "question_answer_short.csv")

In [16]:
query_embedding = model.encode(list(df["question"]))

In [17]:
acceptable_chunks = []
for answer in df["answer"]:
    chunks_ok = set(i for i, chunk in enumerate(chunks) if answer in chunk)
    acceptable_chunks.append(chunks_ok)

In [18]:
def compute_mrr(sim_score, acceptable_chunks):
    ranks = []
    for this_score, this_acceptable_chunks in zip(sim_score, acceptable_chunks):
        indexes = reversed(np.argsort(this_score))
        rank = 1 + next(i for i, idx in enumerate(indexes) if idx in this_acceptable_chunks)
        ranks.append(rank)
        
    return {
        "score": sum(1 / r if r < 6 else 0 for r in ranks) / len(ranks),
        "ranks": ranks,
    }

In [19]:
sim_scores = query_embedding @ corpus_embedding.T

In [20]:
res = compute_mrr(sim_scores, acceptable_chunks)
res["score"]

0.6

# Text generation

In [21]:
def get_context(query, corpus, corpus_embeddings):
    query_embedding = model.encode([query])
    sim_scores = query_embedding @ corpus_embedding.T
    indexes = list(np.argsort(sim_scores[0]))[-5:]
    return [corpus[i] for i in indexes]

In [22]:
get_context("Which class will teach me to build a chatbot?", chunks, corpus_embedding)

['# Natural Language Processing (NLP) Fundamentals and Applications: 5. **Applications of NLP**\n  - Sentiment analysis and text classification\n  - Machine translation and summarization\n  - Chatbots and conversational agents',
 '# Natural Language Processing (NLP) Fundamentals and Applications: **Description:**\nThis course offers a comprehensive introduction to the field of Natural Language Processing (NLP), focusing on the computational techniques that allow machines to understand, interpret, and generate human language. You will learn about linguistic structures, text preprocessing, sentiment analysis, machine translation, and language modeling. Using hands-on projects and industry-relevant tools, this course provides a strong foundation in both traditional and modern NLP methods, including neural networks and transformers.',
 '# Natural Language Processing (NLP) Fundamentals and Applications: Whether you aim to pursue a career in AI or enhance your programming toolkit, this cours

## SMOLL

In [23]:
from transformers import AutoModelForCausalLM, AutoTokenizer

checkpoint = "HuggingFaceTB/SmolLM2-360M-Instruct"
# checkpoint = "HuggingFaceTB/SmolLM2-1.7B-Instruct"
# checkpoint = "amd/Instella-3B"

device = "cpu" # for GPU usage or "cpu" for CPU usage

tokenizer = AutoTokenizer.from_pretrained(checkpoint)
model_generator = AutoModelForCausalLM.from_pretrained(checkpoint).to(device)

tokenizer_config.json:   0%|          | 0.00/3.76k [00:00<?, ?B/s]

In [24]:
def build_smoll_prompt(query, corpus, corpus_embedding):
    context_str = "\n\n".join(get_context(query, chunks, corpus_embedding))

    prompt = f"""<|im_start|>system
You reply to the user's request using only context information.
Context information to answer "{query}" is below
------
Context:
{context_str}
------
You are a helpful assistant for a Computer Science university. You reply to students'questions about the courses that they can attend.
<|im_end|>
<|im_start|>user
{query}
<|im_reend|>
"""
    return prompt


In [25]:
def build_smoll_messages(query, chunks, corpus_embedding):
    context_str = "\n\n".join(get_context(query, chunks, corpus_embedding))

    messages = [
        {"role": "system", "content": f"""You reply to the user's request using only context information.
Context information to answer "{query}" is below
------
Context:
{context_str}
------
You are a helpful assistant for a Computer Science university. You reply to students'questions about the courses that they can attend.
"""},
        {"role": "user", "content": query},
    ]

    return messages


In [29]:
messages = build_smoll_messages("Who is the NLP teacher?", chunks, corpus_embedding)

input_text=tokenizer.apply_chat_template(messages, tokenize=False)
inputs = tokenizer.encode(input_text, return_tensors="pt").to(device)
outputs = model_generator.generate(inputs, max_new_tokens=100, temperature=0.01, top_p=0.9, do_sample=True)
print(tokenizer.decode(outputs[0]))

<|im_start|>system
You reply to the user's request using only context information.
Context information to answer "Who is the NLP teacher?" is below
------
Context:
# Natural Language Processing (NLP) Fundamentals and Applications: **Prerequisites:**
- Proficiency in Python programming
- Basic understanding of linear algebra and probability
- Successful completion of "Introduction to Machine Learning" or equivalent

# Natural Language Processing (NLP) Fundamentals and Applications: **Course Outline:**
1. **Introduction to NLP**
  - Key concepts and challenges
  - Overview of linguistic structure and grammar

# Natural Language Processing (NLP) Fundamentals and Applications: # Natural Language Processing (NLP) Fundamentals and Applications

# Natural Language Processing (NLP) Fundamentals and Applications: **Schedule Time:**
- Tuesdays and Thursdays: 10:00 AM - 11:30 AM
- Lab Sessions: Fridays 2:00 PM - 4:00 PM

# Natural Language Processing (NLP) Fundamentals and Applications: **Teacher

# OpenAI generator
Si vous voulez utiliser OpenAI

In [1]:
openai_key = "YOUR-API-KEY"

In [27]:
import openai

In [28]:
client = openai.OpenAI(api_key=openai_key)

In [30]:
query = "What must I do to pass the NLP class?"

context_str = "\n\n".join(get_context(query, chunks, corpus_embedding))

prompt = f"""Context information is below.
---------------------
{context_str}
---------------------
Given the context information and not prior knowledge, answer the query.
If the answer is not in the context information, reply "I cannot answer that question".
Query: {query}
Answer:"""

In [45]:
res = client.chat.completions.create(                                            
    messages=[{"role": "user", "content": prompt}],                              
    model="gpt-4o-mini",                                                                 
)                                                                                

In [46]:
res.choices[0].message.content

'To pass the NLP class, you need to perform well in the following assessments:\n\n- Complete weekly coding assignments (30% of your grade)\n- Successfully take the midterm exam (20% of your grade)\n- Build an end-to-end NLP application for the final project (30% of your grade)\n- Participate in class discussions and code reviews (20% of your grade)\n\nMake sure to meet the prerequisites, including proficiency in Python, a basic understanding of linear algebra and probability, and having completed "Introduction to Machine Learning" or an equivalent course.'